# Forward PINN for Darcy Flow Below a Reservoir

Goal: build a **forward Physics-Informed Neural Network** for the hydraulic head field `h(x, y)`.

Important distinction:

- The Excel files are useful for **visualizing the problem** and later checking your result.
- The forward PINN loss should use **only the PDE and boundary conditions**, not the measured/generated head values.

This notebook is intentionally written as a scaffold. Some PyTorch pieces are left as `TODO` so you can implement the model yourself.


# 0. Imports

In [ ]:
%pip install -q numpy pandas matplotlib scipy openpyxl torch

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata

import torch
import torch.nn as nn
import random


# Set the same random seed for reproducible runs.
random.seed(1)
np.random.seed(1)
torch.manual_seed(1)

# Prefer CUDA, then Apple Metal (MPS), and otherwise use the CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# 1. Plot data

The filename pattern is:

```text
h1_h2_hd.xlsx
```

where:

- `h1` is the reservoir head,
- `h2` is the downstream catchment head,
- `hd` is the embedded wall depth.

For a forward PINN, we do **not** train on `FINIT`; here we only inspect it so we understand the geometry and have something to compare with after training.

We choose to base ourselves on the dataset
20_5_25.xlsx


In [ ]:
numericldata = pd.read_excel('../Data/heads/20_5_25.xlsx',header=0)
X = numericldata['X'].to_numpy()
Y = numericldata['Y'].to_numpy()
h = numericldata['FINIT'].to_numpy()

fig, ax = plt.subplots()
    
scatter = ax.scatter(X, Y, c=h, cmap='viridis')
plt.colorbar(scatter)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Scatter Plot of Water Dam Data')

### Make it into a continous map of h values

In [ ]:
# Geometry of problem
h1 = 20
h2 = 5
hd = 25

L = 150

y_top = 40 # top of aquifer
y_bot = 0  # bottom of aquifer
x_left = 0  # left boundary of aquifer
x_right = L   # right boundary of aquifer

x_dleft = 100   # left boundary of dam
x_dright = 120  # right boundary of dam
y_dbot = 40 - hd    # bottom of dam

def interpolate_head(X, Y, h, 
                     grid_res=(300, 100),
                     domain_bounds=(x_left, x_right, y_bot, y_top),
                     dam_bounds=(x_dleft, x_dright, y_dbot, y_top),
                     method="linear"):
    
    x_min, x_max, y_min, y_max = domain_bounds
    num_x, num_y = grid_res

    # Generate regular 2D grid
    xi = np.linspace(x_min, x_max, num_x)
    yi = np.linspace(y_min, y_max, num_y)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # Interpolate scattered points
    grid_h = griddata((X,Y), h, (grid_x, grid_y), method=method)

    # Mask out dam region
    x_dl, x_dr, y_db, y_dt = dam_bounds
    dam_mask = (grid_x >= x_dl) & (grid_x <= x_dr) & (grid_y >= y_db) & (grid_y <= y_dt)
    # Replace all h values within the dam area with nan
    grid_h[dam_mask] = np.nan

    return grid_x, grid_y, grid_h


In [ ]:
grid_X, grid_Y, grid_h = interpolate_head(X, Y, h)

fig, ax = plt.subplots()
contour = ax.contourf(grid_X, grid_Y, grid_h, levels=100, cmap="viridis")
cbar = fig.colorbar(contour, ax=ax)
cbar.set_label('Hydraulic head h [m]')
ax.contour(grid_X, grid_Y, grid_h, levels=10, colors='black', linewidths=1, alpha=0.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Interpolated Water Head Field')
plt.show()

## 3. Nondimensionalize the Governing Equation

The dimensional PDE is Laplace's equation:

$$
\Delta h = \frac{\partial^2 h}{\partial x^2} + \frac{\partial^2 h}{\partial y^2} = 0.
$$

Choose one length scale, for example the total horizontal length:

$$
L = 150\;\text{m}.
$$

Define dimensionless coordinates and head:

$$
\hat{x}=\frac{x}{L}, \qquad \hat{y}=\frac{y}{L}, \qquad
H = \frac{h-h_2}{h_1-h_2}.
$$

Then:

$$
h = h_2 + (h_1-h_2)H.
$$

Because `h2` and `(h1-h2)` are constants, the PDE becomes:

$$
\frac{\partial^2 H}{\partial \hat{x}^2} +
\frac{\partial^2 H}{\partial \hat{y}^2}=0.
$$

Boundary heads become simple:

$$
H=1 \quad \text{on the reservoir boundary},
$$

$$
H=0 \quad \text{on the downstream catchment boundary}.
$$

Impermeable boundaries satisfy no-flow, so the normal derivative is zero:

$$
\nabla H \cdot n = 0.
$$


In [ ]:
def to_hat_xy(xy):
    """Convert dimensional coordinates [m] to nondimensional coordinates."""
    return xy / L

def to_hat_head(h):
    return (h-h2)/(h1-h2)

def from_hat_head(H):
    """Convert nondimensional head H back to dimensional head h [m]."""
    return h2 + (h1 - h2) * H

grid_X_hat, grid_Y_hat, grid_h_hat = to_hat_xy(grid_X), to_hat_xy(grid_Y), to_hat_head(grid_h)

fig, ax = plt.subplots()
contour = ax.contourf(grid_X_hat, grid_Y_hat, grid_h_hat, levels=100, cmap="viridis")
cbar = fig.colorbar(contour, ax=ax)
cbar.set_label('Dimensionless head H')
ax.contour(grid_X_hat, grid_Y_hat, grid_h_hat, levels=10, colors='black', linewidths=1, alpha=0.5)
ax.set_xlabel(r'$\hat{x}$')
ax.set_ylabel(r'$\hat{y}$')
ax.set_title('Dimensionless reference head field')
plt.show()

# 2. Draw boundaries and sample collocation and boundary points

Collocation points and boundary condition points. We have three types of boundaries:
1. Boundaries where the value of $h$ is fixed to either $h_1$ or $h_2$
2. Boundaries where $v_x = 0$
3. Boundaries where $v_y = 0$

In [ ]:
# normalise domain and dam bounds
# xmin, xmax, ymin, ymax
domain_bounds = np.array([x_left, x_right, y_bot, y_top])
dam_bounds = np.array([x_dleft, x_dright, y_dbot, y_top])

domain_bounds_norm = to_hat_xy(domain_bounds)
dam_bounds_norm = to_hat_xy(dam_bounds)

print(domain_bounds_norm)
print(dam_bounds_norm)

In [ ]:

def sample_boundaries(n_density, domain_bounds, dam_bounds):

    x_l, x_r, y_b, y_t = domain_bounds
    x_dl, x_dr, y_db, y_dt = dam_bounds


    x1 = np.linspace(x_l, x_dl, int(n_density * (x_dl-x_l)))
    y1 = np.ones(int(n_density * (x_dl-x_l))) * y_t
    h1 = np.column_stack([x1, y1])

    x2 = np.linspace(x_dr, x_r, int(n_density*(x_r-x_dr)))
    y2 = np.ones(int(n_density*(x_r-x_dr))) * y_t
    h2 = np.column_stack([x2, y2])


    x3 = np.ones(int(n_density*(y_t-y_db))) * x_dl
    y3 = np.linspace(y_db, y_t, int(n_density*(y_t-y_db)))

    x4 = np.ones(int(n_density*(y_t-y_db))) * x_dr
    y4 = np.linspace(y_db, y_t, int(n_density*(y_t-y_db)))

    x7 = np.ones(int(n_density*(y_t))) * x_r
    y7 = np.linspace(y_b, y_t, int(n_density*(y_t)))
    y5 = np.ones(int(n_density*(x_dr-x_dl))) * y_db
    x5 = np.linspace(x_dl, x_dr, int(n_density*(x_dr-x_dl)))

    y6 = np.zeros(int(n_density*(x_r)))
    x6 = np.linspace(x_l, x_r, int(n_density*(x_r)))

    x8 = np.ones(int(n_density*(y_t))) * x_l
    y8 = np.linspace(y_b, y_t, int(n_density*(y_t)))

    # All vertical impermeable segments require dH/dx_hat = 0.
    vx0 = np.column_stack([
        np.concatenate([x3, x4, x7, x8]),
        np.concatenate([y3, y4, y7, y8])
    ])

    # Horizontal impermeable segments require dH/dy_hat = 0.
    vy0 = np.column_stack([
        np.concatenate([x5, x6]),
        np.concatenate([y5, y6])
    ])

    return {
        "h1": h1,
        "h2": h2,
        "vx0": vx0,
        "vy0": vy0,
    }

def sample_interior_points(num_points, domain_bounds, dam_bounds):
    
    x_l, x_r, y_b, y_t = domain_bounds
    x_dl, x_dr, y_db, y_dt = dam_bounds

    sampled_x = []
    sampled_y = []
    
    while len(sampled_x) < num_points:
        # 1. Generate a random candidate point inside the global bounding box
        x_candidate = np.random.uniform(x_l, x_r)
        y_candidate = np.random.uniform(y_b, y_t)
        
        # 2. Check if the point falls inside the forbidden dam cutout zone
        is_inside_dam = (x_dl <= x_candidate <= x_dr) and (y_db <= y_candidate <= y_t)
        
        # 3. Only keep the point if it is NOT inside the dam
        if not is_inside_dam:
            sampled_x.append(x_candidate)
            sampled_y.append(y_candidate)
            
    return np.column_stack([np.array(sampled_x), np.array(sampled_y)])

# Sample boundary points
bc_density = 1000
bc = sample_boundaries(bc_density, domain_bounds_norm, dam_bounds_norm)


x_bc_h1, y_bc_h1 = bc['h1'][:, 0].reshape(-1, 1), bc['h1'][:, 1].reshape(-1, 1)
x_bc_h2, y_bc_h2 = bc['h2'][:, 0].reshape(-1, 1), bc['h2'][:, 1].reshape(-1, 1)
x_bc_vx0, y_bc_vx0 = bc["vx0"][:, 0].reshape(-1, 1), bc["vx0"][:, 1].reshape(-1, 1)
x_bc_vy0, y_bc_vy0 = bc["vy0"][:, 0].reshape(-1, 1), bc["vy0"][:, 1].reshape(-1, 1)



h1_color = "orange"
h2_color = "green"
vx0_color = "red"
vy0_color = "blue"


plt.scatter(x_bc_h1, y_bc_h1, color=h1_color, label="$H=1$")
plt.scatter(x_bc_h2, y_bc_h2, color=h2_color, label="$H=0$")
plt.scatter(x_bc_vx0, y_bc_vx0, color=vx0_color, label=r"$H_{,\hat{x}}=0$")
plt.scatter(x_bc_vy0, y_bc_vy0, color=vy0_color, label=r"$H_{,\hat{y}}=0$")
plt.legend()


# Sample interior points
n_points = 10000
xy_internal = sample_interior_points(n_points, domain_bounds_norm, dam_bounds_norm)
x_internal, y_internal = xy_internal[:,0].reshape(-1, 1), xy_internal[:,1].reshape(-1, 1)



plt.scatter(xy_internal[:,0], xy_internal[:,1], color='lightgray', s=10, label='Interior Sampled Points', alpha=0.7, zorder=-2)


plt.show()





# 4. Define NN model

In [ ]:
class MLP(nn.Module):
    # arch = [inputs] + [hidden neurons]*number + [outputs]
    def __init__(self, arch, act):
        super(MLP, self).__init__()
        layers = []
        for i in range(len(arch) - 1):
            layers.append(nn.Linear(arch[i], arch[i + 1]))
            if i < len(arch) - 2:
                layers.append(act())
        self.model = nn.Sequential(*layers)

    def forward(self, inputs):
        out = self.model(inputs)
        return out

# 5. Define PINN Loss functions

Our model receives non-dimensional coordinates and predicts non-dimensional head:

$$
(\hat{x}, \hat{y}) \mapsto H_\theta(\hat{x}, \hat{y}).
$$

We have two types of loss: boundary condition loss and PDE loss. Each boundary type has its own loss:





In [ ]:
# when head must have a specfic numerical value at boundary
def loss_bc_value(model, x, y, val):
    x = torch.tensor(x, dtype=torch.float32, requires_grad=True, device=device)
    y = torch.tensor(y, dtype=torch.float32, requires_grad=True, device=device)

    input = torch.cat((x,y), dim=1)
    H = model(input)

    bc_real = torch.full_like(H, val, device=device)
    bc_loss = torch.mean((H - bc_real)**2)
    return bc_loss

# when one of the derivatives must vanish
# dim = 0 or 1 (x or y)
def loss_bc_grad(model, x, y, dim):
    x = torch.tensor(x, dtype=torch.float32, requires_grad=True, device=device)
    y = torch.tensor(y, dtype=torch.float32, requires_grad=True, device=device)

    input = torch.cat((x,y), dim=1)
    H = model(input)
    
    u = x if dim==0 else y
    H_u = torch.autograd.grad(H, u,
                              grad_outputs=torch.ones_like(H),
                              create_graph=True)[0]
    
    # derivative should vanish, so nonzero value is taken as loss
    bc_loss = torch.mean((H_u)**2)
    return bc_loss

# PDE loss, from LaPlace equation
def loss_pde(model, x, y):
    x = torch.tensor(x, dtype=torch.float32, requires_grad=True, device=device)
    y = torch.tensor(y, dtype=torch.float32, requires_grad=True, device=device)

    input = torch.cat((x,y), dim=1)
    H = model(input)
    H_x = torch.autograd.grad(H, x,
                              grad_outputs=torch.ones_like(H),
                              create_graph=True)[0]
    H_xx = torch.autograd.grad(H_x, x,
                              grad_outputs=torch.ones_like(H),
                              create_graph=True)[0]
    H_y = torch.autograd.grad(H, y,
                              grad_outputs=torch.ones_like(H),
                              create_graph=True)[0]
    H_yy = torch.autograd.grad(H_y, y,
                              grad_outputs=torch.ones_like(H),
                              create_graph=True)[0]

    pde_residual = H_xx + H_yy
    return torch.mean(pde_residual**2)


# when y derivative of head must vanish



# 6. Fixed-weight PINN loss and training

The network maps nondimensional coordinates to nondimensional head:

$$ (\hat{x},\hat{y}) \mapsto H_\theta(\hat{x},\hat{y}). $$

The grouped loss is

$$ \mathcal{L}=\mathcal{L}_{PDE}+\lambda_D\mathcal{L}_D+\lambda_N\mathcal{L}_N, $$

where

$$ \mathcal{L}_{PDE}=\operatorname{MSE}(H_{,\hat{x}\hat{x}}+H_{,\hat{y}\hat{y}},0), $$

$$ \mathcal{L}_D=\operatorname{MSE}(H,1)_{reservoir}+\operatorname{MSE}(H,0)_{catchment}, $$

$$ \mathcal{L}_N=\operatorname{MSE}(H_{,\hat{x}},0)_{vertical}+\operatorname{MSE}(H_{,\hat{y}},0)_{horizontal}. $$


In [ ]:
# Create neural network with tanh activation function
layers = [2] + [50]*4 + [1]
model = MLP(layers, nn.Tanh).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Number of epochs
num_epochs = 1000

lambda_D = 20.0
lambda_N = 5.0

history_fixed = {
    "total": [], "pde": [], "dirichlet": [], "neumann": [],
    "bc_h1": [], "bc_h2": [], "bc_vx0": [], "bc_vy0": [],
}

# Training loop
for epoch in range(num_epochs):
    optimizer.zero_grad()
    # Compute losses
    bc_h1_loss = loss_bc_value(model, bc['h1'][:,0].reshape(-1, 1), bc['h1'][:,1].reshape(-1, 1), 1)
    bc_h2_loss = loss_bc_value(model, bc['h2'][:,0].reshape(-1, 1), bc['h2'][:,1].reshape(-1, 1), 0)
    bc_vx0_loss = loss_bc_grad(model, bc['vx0'][:,0].reshape(-1, 1), bc['vx0'][:,1].reshape(-1, 1), dim=0)
    bc_vy0_loss = loss_bc_grad(model, bc['vy0'][:,0].reshape(-1, 1), bc['vy0'][:,1].reshape(-1, 1), dim=1)

    pde_loss = loss_pde(model, x_internal, y_internal)

    dirichlet_loss = bc_h1_loss + bc_h2_loss
    neumann_loss = bc_vx0_loss + bc_vy0_loss
    loss = pde_loss + lambda_D * dirichlet_loss + lambda_N * neumann_loss
    loss.backward()
    optimizer.step()

    history_fixed["total"].append(loss.item())
    history_fixed["pde"].append(pde_loss.item())
    history_fixed["dirichlet"].append(dirichlet_loss.item())
    history_fixed["neumann"].append(neumann_loss.item())
    history_fixed["bc_h1"].append(bc_h1_loss.item())
    history_fixed["bc_h2"].append(bc_h2_loss.item())
    history_fixed["bc_vx0"].append(bc_vx0_loss.item())
    history_fixed["bc_vy0"].append(bc_vy0_loss.item())
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4e}, PDE Loss = {pde_loss.item():.4e}, BC h1 Loss = {bc_h1_loss.item():.4e}, BC h2 Loss {bc_h2_loss.item():.4e}, BC vx0 Loss {bc_vx0_loss}, BC vy0 Loss {bc_vy0_loss}")


# Keep a stable reference before creating the adaptive model.
model_fixed = model

epochs = np.arange(1, len(history_fixed["total"]) + 1)
plt.semilogy(epochs, history_fixed["total"])
plt.ylabel("Loss")
plt.xlabel("Epoch")

# 7. Prediction helpers

The model is evaluated with nondimensional coordinates and its output is converted back to dimensional hydraulic head for plotting.

In [ ]:
def predict_head_grid(model, nx=300, ny=100):
    """Evaluate dimensionless H on the aquifer grid and return dimensional h."""
    x_grid = np.linspace(x_left, x_right, nx)
    y_grid = np.linspace(y_bot, y_top, ny)
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid, indexing="xy")
    xy_dim = np.column_stack([x_mesh.ravel(), y_mesh.ravel()])
    xy_hat = to_hat_xy(xy_dim)

    model.eval()
    with torch.no_grad():
        H_pred = model(torch.as_tensor(xy_hat, dtype=torch.float32, device=device))

    H_pred = H_pred.detach().cpu().numpy().reshape(y_mesh.shape)
    h_pred = from_hat_head(H_pred)
    dam_mask = ((x_mesh >= x_dleft) & (x_mesh <= x_dright) &
                (y_mesh >= y_dbot) & (y_mesh <= y_top))
    return x_mesh, y_mesh, np.ma.masked_where(dam_mask, h_pred)

def plot_head_prediction(model, title, nx=300, ny=100):
    """Plot predicted dimensional head on physical coordinates."""
    x_mesh, y_mesh, h_pred = predict_head_grid(model, nx=nx, ny=ny)

    fig, ax = plt.subplots(figsize=(10, 4))
    levels = np.linspace(h2, h1, 101)
    contour = ax.contourf(
        x_mesh, y_mesh, h_pred, levels=levels, cmap="viridis", extend="both"
    )
    ax.contour(
        x_mesh, y_mesh, h_pred, levels=np.linspace(h2, h1, 11),
        colors="white", linewidths=0.6, alpha=0.55
    )
    ax.fill(
        [x_dleft, x_dright, x_dright, x_dleft],
        [y_dbot, y_dbot, y_top, y_top],
        color="0.6", label="impermeable dam"
    )
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label("Predicted hydraulic head h [m]")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(title)
    ax.set_xlim(x_left, x_right)
    ax.set_ylim(y_bot, y_top)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="lower left")
    plt.tight_layout()
    plt.show()
    return fig, ax, h_pred

In [ ]:
# Fixed-weight baseline prediction (before adaptive-weight retraining).
plot_head_prediction(model, "Fixed-weight PINN prediction")

# 8. Adaptive weights (NTK-based)

This section adapts the Day-5 NTK idea to the Darcy/Laplace PINN.

For each loss component, a residual vector is formed:

- reservoir Dirichlet boundary: $H-1$
- downstream Dirichlet boundary: $H$
- vertical-wall/no-flow boundaries: $H_x$
- horizontal/no-flow boundaries: $H_y$
- PDE residual: $H_{xx}+H_{yy}$

The average trace of the empirical Neural Tangent Kernel is estimated from a
small residual batch. Loss terms with a small NTK trace receive a larger weight.
An exponential moving average is used so that the weights do not jump abruptly.
The adaptive component weights remain inside the same grouped objective: the two Dirichlet components are multiplied by $\lambda_D$, the two no-flow components by $\lambda_N$, and the PDE component remains unscaled.


In [ ]:
class DarcyAdaptiveWeights:
    """NTK-based adaptive loss weighting for the Darcy/Laplace PINN."""

    names = ("bc_h1", "bc_h2", "bc_vx0", "bc_vy0", "pde")

    def __init__(self, model, device, batch_size=32, ema=0.9, eps=1e-12):
        self.model = model
        self.device = device
        self.batch_size = batch_size
        self.ema = ema
        self.eps = eps
        self.weights = {name: 1.0 for name in self.names}

    @staticmethod
    def _tensor(values, device, requires_grad=True):
        return torch.as_tensor(
            values, dtype=torch.float32, device=device
        ).reshape(-1, 1).requires_grad_(requires_grad)

    def residual_bc_value(self, xy, target):
        x = self._tensor(xy[:, 0], self.device)
        y = self._tensor(xy[:, 1], self.device)
        H = self.model(torch.cat((x, y), dim=1))
        return H - target

    def residual_bc_grad(self, xy, dim):
        x = self._tensor(xy[:, 0], self.device)
        y = self._tensor(xy[:, 1], self.device)
        H = self.model(torch.cat((x, y), dim=1))
        coordinate = x if dim == 0 else y
        return torch.autograd.grad(
            H,
            coordinate,
            grad_outputs=torch.ones_like(H),
            create_graph=True,
        )[0]

    def residual_pde(self, xy):
        x = self._tensor(xy[:, 0], self.device)
        y = self._tensor(xy[:, 1], self.device)
        H = self.model(torch.cat((x, y), dim=1))

        H_x = torch.autograd.grad(
            H, x, grad_outputs=torch.ones_like(H), create_graph=True
        )[0]
        H_y = torch.autograd.grad(
            H, y, grad_outputs=torch.ones_like(H), create_graph=True
        )[0]
        H_xx = torch.autograd.grad(
            H_x, x, grad_outputs=torch.ones_like(H_x), create_graph=True
        )[0]
        H_yy = torch.autograd.grad(
            H_y, y, grad_outputs=torch.ones_like(H_y), create_graph=True
        )[0]
        return H_xx + H_yy

    def residuals(self, bc, xy_internal):
        return {
            "bc_h1": self.residual_bc_value(bc["h1"], 1.0),
            "bc_h2": self.residual_bc_value(bc["h2"], 0.0),
            "bc_vx0": self.residual_bc_grad(bc["vx0"], dim=0),
            "bc_vy0": self.residual_bc_grad(bc["vy0"], dim=1),
            "pde": self.residual_pde(xy_internal),
        }

    @staticmethod
    def mse_losses(residuals):
        return {name: torch.mean(res**2) for name, res in residuals.items()}

    def _sample_residual(self, residual):
        n = residual.shape[0]
        if n <= self.batch_size:
            return residual
        idx = torch.randperm(n, device=residual.device)[:self.batch_size]
        return residual[idx]

    def _mean_ntk_trace(self, residual):
        """
        Estimate trace(J J^T) / N without explicitly constructing the full NTK.
        A small residual batch keeps memory use manageable.
        """
        residual = self._sample_residual(residual).reshape(-1)
        params = [p for p in self.model.parameters() if p.requires_grad]

        trace = torch.zeros((), device=self.device)
        for i in range(residual.numel()):
            grads = torch.autograd.grad(
                residual[i],
                params,
                retain_graph=True,
                create_graph=False,
                allow_unused=True,
            )

            # Derivative-based residuals do not depend on every parameter.
            # For example, the output bias disappears in H_x, H_y, H_xx,
            # and H_yy. PyTorch therefore returns None for those parameters.
            trace = trace + sum(
                torch.sum(g.detach() ** 2)
                for g in grads
                if g is not None
            )

        return (trace / residual.numel()).item()

    def update(self, residuals):
        traces = {
            name: max(self._mean_ntk_trace(residuals[name]), self.eps)
            for name in self.names
        }

        trace_sum = sum(traces.values())
        raw_weights = {
            name: trace_sum / traces[name]
            for name in self.names
        }

        # Keep the average weight equal to one so the overall loss scale
        # remains comparable to the unweighted formulation.
        raw_mean = sum(raw_weights.values()) / len(raw_weights)
        raw_weights = {
            name: value / raw_mean
            for name, value in raw_weights.items()
        }

        self.weights = {
            name: self.ema * self.weights[name]
            + (1.0 - self.ema) * raw_weights[name]
            for name in self.names
        }
        return self.weights.copy(), traces


# Fresh model for adaptive-weight training
layers = [2] + [50] * 4 + [1]
model_aw = MLP(layers, nn.Tanh).to(device)

optimizer_aw = torch.optim.Adam(model_aw.parameters(), lr=1e-3)
scheduler_aw = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_aw, mode="min", factor=0.5, patience=300
)

num_epochs_aw = 5000
weight_update_every = 50
print_every = 100

adaptive = DarcyAdaptiveWeights(
    model=model_aw,
    device=device,
    batch_size=32,
    ema=0.9,
)

history_aw = {
    "total": [],
    **{name: [] for name in adaptive.names},
    **{f"w_{name}": [] for name in adaptive.names},
}

for epoch in range(num_epochs_aw):
    optimizer_aw.zero_grad()

    residuals = adaptive.residuals(bc, xy_internal)
    component_losses = adaptive.mse_losses(residuals)

    # Update NTK weights periodically. At epoch 0 this initializes them
    # from the current network gradients.
    if epoch % weight_update_every == 0:
        current_weights, current_traces = adaptive.update(residuals)

    # Retain NTK weights within the PDE/Dirichlet/Neumann grouping.
    weighted_pde = adaptive.weights["pde"] * component_losses["pde"]
    weighted_dirichlet = (
        adaptive.weights["bc_h1"] * component_losses["bc_h1"]
        + adaptive.weights["bc_h2"] * component_losses["bc_h2"]
    )
    weighted_neumann = (
        adaptive.weights["bc_vx0"] * component_losses["bc_vx0"]
        + adaptive.weights["bc_vy0"] * component_losses["bc_vy0"]
    )
    total_loss = (
        weighted_pde + lambda_D * weighted_dirichlet + lambda_N * weighted_neumann
    )

    total_loss.backward()
    optimizer_aw.step()
    scheduler_aw.step(total_loss.item())

    history_aw["total"].append(total_loss.item())
    for name in adaptive.names:
        history_aw[name].append(component_losses[name].item())
        history_aw[f"w_{name}"].append(adaptive.weights[name])

    if epoch % print_every == 0:
        loss_text = ", ".join(
            f"{name}={component_losses[name].item():.2e}"
            for name in adaptive.names
        )
        weight_text = ", ".join(
            f"w_{name}={adaptive.weights[name]:.2f}"
            for name in adaptive.names
        )
        print(
            f"Epoch {epoch:5d} | total={total_loss.item():.3e} | "
            f"{loss_text} | {weight_text}"
        )

# Preserve the adaptive model as the active model and visualize its prediction.
model = model_aw
plot_head_prediction(
    model_aw,
    "Adaptive-weight PINN prediction (NTK weighting)"
)


In [ ]:
# Plot total/component losses and the learned adaptive weights
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].semilogy(history_aw["total"], label="total")
for name in adaptive.names:
    axes[0].semilogy(history_aw[name], label=name, alpha=0.8)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("Adaptive-weight PINN losses")
axes[0].legend()
axes[0].grid(True)

for name in adaptive.names:
    axes[1].plot(history_aw[f"w_{name}"], label=f"w_{name}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Weight")
axes[1].set_title("NTK-based adaptive weights")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Compare reference data, fixed-weight PINN, and adaptive-weight PINN.
def reference_metrics(model):
    xy_ref = numericldata[["X", "Y"]].to_numpy(dtype=float)
    h_true = numericldata["FINIT"].to_numpy(dtype=float)
    model.eval()
    with torch.no_grad():
        H_pred = model(torch.as_tensor(
            to_hat_xy(xy_ref), dtype=torch.float32, device=device
        )).detach().cpu().numpy().ravel()
    h_pred = from_hat_head(H_pred)
    error = h_pred - h_true
    return float(np.sqrt(np.mean(error**2))), float(np.mean(np.abs(error)))

x_fixed, y_fixed, h_fixed = predict_head_grid(model_fixed)
x_adaptive, y_adaptive, h_adaptive = predict_head_grid(model_aw)
h_reference = np.ma.masked_invalid(grid_h)
rmse_fixed, mae_fixed = reference_metrics(model_fixed)
rmse_adaptive, mae_adaptive = reference_metrics(model_aw)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
levels = np.linspace(h2, h1, 101)
fields = [h_reference, h_fixed, h_adaptive]
meshes = [(grid_X, grid_Y), (x_fixed, y_fixed), (x_adaptive, y_adaptive)]
titles = [
    "Reference head field",
    f"Fixed weights\nRMSE={rmse_fixed:.3f} m, MAE={mae_fixed:.3f} m",
    f"Adaptive NTK weights\nRMSE={rmse_adaptive:.3f} m, MAE={mae_adaptive:.3f} m",
]

for ax, (xm, ym), field, title in zip(axes, meshes, fields, titles):
    contour = ax.contourf(
        xm, ym, field, levels=levels, cmap="viridis", extend="both"
    )
    ax.contour(
        xm, ym, field, levels=np.linspace(h2, h1, 11),
        colors="white", linewidths=0.45, alpha=0.5
    )
    ax.fill(
        [x_dleft, x_dright, x_dright, x_dleft],
        [y_dbot, y_dbot, y_top, y_top], color="0.6"
    )
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_xlim(x_left, x_right)
    ax.set_ylim(y_bot, y_top)
    ax.set_aspect("equal", adjustable="box")
axes[0].set_ylabel("y [m]")
fig.colorbar(contour, ax=axes, label="Hydraulic head h [m]", shrink=0.9)
plt.show()


In [ ]:
# Export timestamped summaries for fixed- and adaptive-weight runs.
from datetime import datetime
from pathlib import Path
import json

# Locate the project root whether the notebook kernel starts in Code/ or PIIM/.
working_dir = Path.cwd().resolve()
if (working_dir / "Data").is_dir():
    project_root = working_dir
elif (working_dir.parent / "Data").is_dir():
    project_root = working_dir.parent
else:
    raise FileNotFoundError("Could not locate the project root containing Data/.")

# Evaluate both models at the spreadsheet coordinates.
reference_xy = numericldata[["X", "Y"]].to_numpy(dtype=float)
reference_h = numericldata["FINIT"].to_numpy(dtype=float)
reference_xy_hat = to_hat_xy(reference_xy)

def model_metrics(trained_model):
    trained_model.eval()
    with torch.no_grad():
        H_pred = trained_model(
            torch.as_tensor(reference_xy_hat, dtype=torch.float32, device=device)
        ).detach().cpu().numpy().ravel()
    h_pred = from_hat_head(H_pred)
    valid = np.isfinite(reference_h) & np.isfinite(h_pred)
    error = h_pred[valid] - reference_h[valid]
    ss_res = float(np.sum(error**2))
    ss_tot = float(np.sum((reference_h[valid] - np.mean(reference_h[valid]))**2))
    return {
        "reference_points": int(valid.sum()),
        "rmse_m": float(np.sqrt(np.mean(error**2))),
        "mae_m": float(np.mean(np.abs(error))),
        "max_abs_error_m": float(np.max(np.abs(error))),
        "relative_l2_error": float(np.linalg.norm(error) / np.linalg.norm(reference_h[valid])),
        "r2": float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
        "prediction_min_m": float(np.min(h_pred[valid])),
        "prediction_max_m": float(np.max(h_pred[valid])),
    }

fixed_metrics = model_metrics(model_fixed)
adaptive_metrics = model_metrics(model_aw)

now = datetime.now().astimezone()
date_folder = now.strftime("%Y-%m-%d")
file_stamp = now.strftime("%Y-%m-%d_%H-%M-%S-%f")
results_dir = project_root / "Results" / date_folder
results_dir.mkdir(parents=True, exist_ok=True)
csv_path = results_dir / f"pinn_fixed_vs_adaptive_{file_stamp}.csv"

parameter_count = sum(p.numel() for p in model_aw.parameters())
completed_epochs = len(history_aw["total"])
adaptive_summary = {
    "timestamp": now.isoformat(timespec="seconds"),
    "model_variant": "adaptive_ntk",
    "model_type": type(model_aw).__name__,
    "architecture": json.dumps(layers),
    "activation": "Tanh",
    "parameter_count": parameter_count,
    "weighting_method": "inverse NTK trace with EMA",
    "lambda_D": lambda_D,
    "lambda_N": lambda_N,
    "adaptive_ema": adaptive.ema,
    "ntk_batch_size": adaptive.batch_size,
    "weight_update_every": weight_update_every,
    "epochs_requested": num_epochs_aw,
    "epochs_completed": completed_epochs,
    "learning_rate_final": optimizer_aw.param_groups[0]["lr"],
    "device": str(device),
    "random_seed": 1,
    "interior_points": len(xy_internal),
    "boundary_density": bc_density,
    "h1_m": h1,
    "h2_m": h2,
    "dam_depth_m": hd,
    "domain_length_m": L,
    "final_total_loss": history_aw["total"][-1],
}
adaptive_summary.update(adaptive_metrics)
adaptive_summary["final_loss_dirichlet"] = (
    history_aw["bc_h1"][-1] + history_aw["bc_h2"][-1]
)
adaptive_summary["final_loss_neumann"] = (
    history_aw["bc_vx0"][-1] + history_aw["bc_vy0"][-1]
)

for name in adaptive.names:
    adaptive_summary[f"final_loss_{name}"] = history_aw[name][-1]
    adaptive_summary[f"final_weight_{name}"] = adaptive.weights[name]
    if "current_traces" in globals():
        adaptive_summary[f"final_ntk_trace_{name}"] = current_traces[name]

fixed_summary = {
    **adaptive_summary,
    "model_variant": "fixed_weights",
    "weighting_method": "grouped fixed weights",
    "adaptive_ema": np.nan,
    "ntk_batch_size": np.nan,
    "weight_update_every": np.nan,
    "epochs_requested": num_epochs,
    "epochs_completed": len(history_fixed["total"]),
    "learning_rate_final": optimizer.param_groups[0]["lr"],
    "final_total_loss": history_fixed["total"][-1],
    "final_loss_pde": history_fixed["pde"][-1],
    "final_loss_dirichlet": history_fixed["dirichlet"][-1],
    "final_loss_neumann": history_fixed["neumann"][-1],
    "final_loss_bc_h1": history_fixed["bc_h1"][-1],
    "final_loss_bc_h2": history_fixed["bc_h2"][-1],
    "final_loss_bc_vx0": history_fixed["bc_vx0"][-1],
    "final_loss_bc_vy0": history_fixed["bc_vy0"][-1],
}
fixed_summary.update(fixed_metrics)
for name in adaptive.names:
    fixed_summary[f"final_weight_{name}"] = np.nan
    fixed_summary[f"final_ntk_trace_{name}"] = np.nan

results_table = pd.DataFrame([fixed_summary, adaptive_summary])
results_table.to_csv(csv_path, index=False)
print(f"Saved run summary to: {csv_path}")
print(results_table[["model_variant", "rmse_m", "mae_m", "r2"]])
results_table
